# 06 · Decision Framework

The guide, in one place. Use **LoRA** (freeze the base, train adapters) for adapting
behavior, style, format, domain phrasing, or a task head — which is most fine-tuning.
Pick the rank by sweeping and watching the plateau (nb 04), not by habit. Adapt the
attention and feed-forward projections; you rarely need everything. **Merge** for
zero-overhead inference, or keep adapters swappable for multi-task serving. And choose
the tool by *what you are changing*.

In [1]:
import sys, pathlib
_p = pathlib.Path.cwd()
while not (_p / 'pyproject.toml').exists() and _p != _p.parent:
    _p = _p.parent
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))
print('repo root:', _p)

repo root: /workspaces/lora_101


In [2]:
from src.results import save_json

decision = [
    {'change': 'behavior / style / format / tone', 'use': 'LoRA', 'rank': 'sweep to the plateau',
     'why': 'a low-rank adjustment'},
    {'change': 'a task head / classification', 'use': 'LoRA (or linear probe)', 'rank': 'small',
     'why': 'small, low-rank'},
    {'change': 'domain phrasing', 'use': 'LoRA', 'rank': 'small-medium', 'why': 'low-rank'},
    {'change': 'inject new facts', 'use': 'retrieval (rag_101)', 'rank': 'n/a',
     'why': 'facts are not a low-rank change'},
    {'change': "the whole model's capability", 'use': 'full fine-tune', 'rank': 'n/a',
     'why': 'needs full-rank capacity'},
    {'change': 'fit a big model in little memory', 'use': 'QLoRA (quantization_101 + LoRA)',
     'rank': 'small', 'why': 'quantize the frozen base, adapt with LoRA'},
]
path = save_json({
    'decision_matrix': decision,
    'deployment': {'merge': 'fold adapter for zero-overhead inference',
                   'keep_separate': 'swap adapters for multi-task serving'},
    'rank_rule': "accuracy rises then plateaus at the update's true rank; pick the knee",
}, '06_decision_framework')
print('saved:', path.name)
for row in decision:
    print(f"- change {row['change']!r} -> {row['use']}")

saved: 06_decision_framework.json
- change 'behavior / style / format / tone' -> LoRA
- change 'a task head / classification' -> LoRA (or linear probe)
- change 'domain phrasing' -> LoRA
- change 'inject new facts' -> retrieval (rag_101)
- change "the whole model's capability" -> full fine-tune
- change 'fit a big model in little memory' -> QLoRA (quantization_101 + LoRA)


## A suitability map (PNG)

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from src.visualisation import save_figure

tools = ['LoRA', 'full FT', 'retrieval', 'QLoRA']
uses = ['style / behavior', 'new capability', 'new facts', 'big model, low memory']
m = np.array([
    [1.0, 0.4, 0.1, 0.9],
    [0.5, 1.0, 0.1, 0.4],
    [0.1, 0.3, 1.0, 0.1],
    [0.8, 0.2, 0.1, 1.0],
])
fig, ax = plt.subplots(figsize=(6.5, 4))
im = ax.imshow(m, cmap='Greens', vmin=0, vmax=1)
ax.set_xticks(range(len(tools))); ax.set_xticklabels(tools, rotation=20, ha='right')
ax.set_yticks(range(len(uses))); ax.set_yticklabels(uses)
for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        ax.text(j, i, f'{m[i, j]:.1f}', ha='center', va='center')
ax.set_title('What to use for what you are changing')
fig.colorbar(im, label='suitability')
png = save_figure(fig, '06_decision_matrix')
print('saved:', png.name)

saved: 06_decision_matrix.png


## The close
A new task adds far less information to a model than its parameter count suggests, so
you were fine-tuning millions of weights to learn what fits in a thousand. Use LoRA
for behavior, retrieval for facts, pick the rank from evidence, and merge for
deployment. The from-scratch layer, injection, merge, and accounting all live in
`src/lora/` — working code you can take with you.